In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from pathlib import Path

In [2]:
import time
from pathlib import Path

import soundfile as sf
import torch
from tqdm import tqdm
from torch import multiprocessing

import sys

In [3]:
stored_files = list(Path('../../Documents/mila-human-wav-txt').glob('*.WAV'))
stored_files

[PosixPath('../../Documents/mila-human-wav-txt/20220826_070000.WAV'),
 PosixPath('../../Documents/mila-human-wav-txt/20220727_083000.WAV'),
 PosixPath('../../Documents/mila-human-wav-txt/20220829_090000.WAV'),
 PosixPath('../../Documents/mila-human-wav-txt/20220730_053000.WAV')]

In [5]:
import batdetect2.api as api

Used `python3 -m pip install batdetect2==1.1.1`

Output: `Successfully installed batdetect2-1.1.1`

In [6]:
from batdetect2.detector.parameters import (
    DEFAULT_MODEL_PATH,
    DEFAULT_PROCESSING_CONFIGURATIONS,
    DEFAULT_SPECTROGRAM_PARAMETERS,
    TARGET_SAMPLERATE_HZ,
)

In [7]:
TARGET_SAMPLERATE_HZ

256000

In [8]:
DEFAULT_SPECTROGRAM_PARAMETERS

{'fft_win_length': 0.002,
 'fft_overlap': 0.75,
 'spec_height': 256,
 'resize_factor': 0.5,
 'spec_divide_factor': 32,
 'max_freq': 120000,
 'min_freq': 10000,
 'spec_scale': 'pcen',
 'denoise_spec_avg': True,
 'max_scale_spec': False}

In [9]:
DEFAULT_PROCESSING_CONFIGURATIONS

{'detection_threshold': 0.01,
 'spec_slices': False,
 'chunk_size': 3,
 'spec_features': False,
 'cnn_features': False,
 'quiet': True,
 'target_samp_rate': 256000,
 'fft_win_length': 0.002,
 'fft_overlap': 0.75,
 'resize_factor': 0.5,
 'spec_divide_factor': 32,
 'spec_height': 256,
 'scale_raw_audio': False,
 'class_names': [],
 'time_expansion': 1,
 'top_n': 3,
 'return_raw_preds': False,
 'max_duration': None,
 'nms_kernel_size': 9,
 'max_freq': 120000,
 'min_freq': 10000,
 'nms_top_k_per_sec': 200,
 'spec_scale': 'pcen',
 'denoise_spec_avg': True,
 'max_scale_spec': False}

In [10]:
# Use GPU if available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Default model
MODEL, PARAMS = api.load_model(DEFAULT_MODEL_PATH, device=DEVICE)

In [11]:
PARAMS

{'data_dir': '/data1/bat_data/data/',
 'ann_dir': '/data1/bat_data/annotations/anns_same/',
 'train_split': 'same',
 'standardize_classs_names_ip': 'Rhinolophus ferrumequinum;Rhinolophus hipposideros',
 'model_name': 'Net2DFast',
 'num_filters': 128,
 'experiment': '../../experiments/2021_12_13__20_20_37/',
 'model_file_name': '../../experiments/2021_12_13__20_20_37/2021_12_13__20_20_37.pth.tar',
 'op_im_dir': '../../experiments/2021_12_13__20_20_37/op_ims/',
 'op_im_dir_test': '../../experiments/2021_12_13__20_20_37/op_ims_test/',
 'notes': '',
 'target_samp_rate': 256000,
 'fft_win_length': 0.002,
 'fft_overlap': 0.75,
 'max_freq': 120000,
 'min_freq': 10000,
 'resize_factor': 0.5,
 'spec_height': 256,
 'spec_train_width': 512,
 'spec_divide_factor': 32,
 'denoise_spec_avg': True,
 'scale_raw_audio': False,
 'max_scale_spec': False,
 'spec_scale': 'pcen',
 'detection_overlap': 0.01,
 'ignore_start_end': 0.01,
 'detection_threshold': 0.01,
 'nms_kernel_size': 9,
 'nms_top_k_per_sec': 

In [12]:
# Default processing configuration
ubna_config = api.get_config(detection_threshold=0.5)

In [12]:
pd.DataFrame([ubna_config]).iloc[:,:10]

,detection_threshold,spec_slices,chunk_size,spec_features,cnn_features,quiet,target_samp_rate,fft_win_length,fft_overlap,resize_factor
0,0.5,False,3,False,False,True,256000,0.002,0.75,0.5


In [13]:
pd.DataFrame([ubna_config]).iloc[:,10:20]

,spec_divide_factor,spec_height,scale_raw_audio,class_names,time_expansion,top_n,return_raw_preds,max_duration,nms_kernel_size,max_freq
0,32,256,False,"[Barbastellus barbastellus, Eptesicus serotinu...",1,3,False,None,9,120000


In [14]:
pd.DataFrame([ubna_config]).iloc[:,20:30]

,min_freq,nms_top_k_per_sec,spec_scale,denoise_spec_avg,max_scale_spec,data_dir,ann_dir,train_split,standardize_classs_names_ip,model_name
0,10000,200,pcen,True,False,/data1/bat_data/data/,/data1/bat_data/annotations/anns_same/,same,Rhinolophus ferrumequinum;Rhinolophus hipposid...,Net2DFast


In [15]:
pd.DataFrame([ubna_config]).iloc[:,30:40]

,num_filters,experiment,model_file_name,op_im_dir,op_im_dir_test,notes,spec_train_width,detection_overlap,ignore_start_end,target_sigma
0,128,../../experiments/2021_12_13__20_20_37/,../../experiments/2021_12_13__20_20_37/2021_12...,../../experiments/2021_12_13__20_20_37/op_ims/,../../experiments/2021_12_13__20_20_37/op_ims_...,,512,0.01,0.01,2.0


In [16]:
pd.DataFrame([ubna_config]).iloc[:,40:50]

,aug_prob,augment_at_train,augment_at_train_combine,echo_max_delay,stretch_squeeze_delta,mask_max_time_perc,mask_max_freq_perc,spec_amp_scaling,aug_sampling_rates,train_loss
0,0.2,True,True,0.005,0.04,0.05,0.1,2.0,"[220500, 256000, 300000, 312500, 384000, 44100...",focal


In [17]:
pd.DataFrame([ubna_config]).iloc[:,50:60]

,det_loss_weight,size_loss_weight,class_loss_weight,individual_loss_weight,emb_dim,lr,batch_size,num_workers,num_epochs,num_eval_epochs
0,1.0,0.1,2.0,0.0,0,0.001,8,4,200,5


In [18]:
# Process audio file
results = api.process_file(stored_files[0], config=ubna_config)

In [19]:
results['pred_dict']['annotation']

[{'start_time': 3.7705,
  'end_time': 3.7774,
  'low_freq': 46093,
  'high_freq': 54897,
  'class': 'Pipistrellus pipistrellus',
  'class_prob': 0.505,
  'det_prob': 0.541,
  'individual': '-1',
  'event': 'Echolocation'},
 {'start_time': 4.3085,
  'end_time': 4.3139,
  'low_freq': 46093,
  'high_freq': 58971,
  'class': 'Pipistrellus pipistrellus',
  'class_prob': 0.512,
  'det_prob': 0.542,
  'individual': '-1',
  'event': 'Echolocation'},
 {'start_time': 5.0035,
  'end_time': 5.0094,
  'low_freq': 45234,
  'high_freq': 60411,
  'class': 'Pipistrellus pipistrellus',
  'class_prob': 0.491,
  'det_prob': 0.544,
  'individual': '-1',
  'event': 'Echolocation'},
 {'start_time': 5.1515,
  'end_time': 5.1568,
  'low_freq': 46093,
  'high_freq': 61553,
  'class': 'Pipistrellus pipistrellus',
  'class_prob': 0.443,
  'det_prob': 0.519,
  'individual': '-1',
  'event': 'Echolocation'},
 {'start_time': 5.2725,
  'end_time': 5.2776,
  'low_freq': 46093,
  'high_freq': 61355,
  'class': 'Pipistr

In [20]:
pd.DataFrame(results['pred_dict']['annotation'])

,start_time,end_time,low_freq,high_freq,class,class_prob,det_prob,individual,event
0,3.7705,3.7774,46093,54897,Pipistrellus pipistrellus,0.505,0.541,-1,Echolocation
1,4.3085,4.3139,46093,58971,Pipistrellus pipistrellus,0.512,0.542,-1,Echolocation
2,5.0035,5.0094,45234,60411,Pipistrellus pipistrellus,0.491,0.544,-1,Echolocation
3,5.1515,5.1568,46093,61553,Pipistrellus pipistrellus,0.443,0.519,-1,Echolocation
4,5.2725,5.2776,46093,61355,Pipistrellus pipistrellus,0.557,0.627,-1,Echolocation
...,...,...,...,...,...,...,...,...,...
1571,1787.7745,1787.7848,37500,46349,Pipistrellus nathusii,0.447,0.562,-1,Echolocation
1572,1787.9595,1787.9694,37500,45423,Pipistrellus nathusii,0.517,0.582,-1,Echolocation
1573,1788.2125,1788.2211,39218,47828,Pipistrellus nathusii,0.433,0.557,-1,Echolocation
1574,1788.4146,1788.4226,40078,49782,Pipistrellus nathusii,0.504,0.583,-1,Echolocation


In [21]:
pd.DataFrame.from_records(results['pred_dict']['annotation']) 

,start_time,end_time,low_freq,high_freq,class,class_prob,det_prob,individual,event
0,3.7705,3.7774,46093,54897,Pipistrellus pipistrellus,0.505,0.541,-1,Echolocation
1,4.3085,4.3139,46093,58971,Pipistrellus pipistrellus,0.512,0.542,-1,Echolocation
2,5.0035,5.0094,45234,60411,Pipistrellus pipistrellus,0.491,0.544,-1,Echolocation
3,5.1515,5.1568,46093,61553,Pipistrellus pipistrellus,0.443,0.519,-1,Echolocation
4,5.2725,5.2776,46093,61355,Pipistrellus pipistrellus,0.557,0.627,-1,Echolocation
...,...,...,...,...,...,...,...,...,...
1571,1787.7745,1787.7848,37500,46349,Pipistrellus nathusii,0.447,0.562,-1,Echolocation
1572,1787.9595,1787.9694,37500,45423,Pipistrellus nathusii,0.517,0.582,-1,Echolocation
1573,1788.2125,1788.2211,39218,47828,Pipistrellus nathusii,0.433,0.557,-1,Echolocation
1574,1788.4146,1788.4226,40078,49782,Pipistrellus nathusii,0.504,0.583,-1,Echolocation
